In [29]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Tiền xử lý

In [30]:
import pandas as pd
df = pd.read_excel('/content/drive/MyDrive/SE363-Team13/data/19_type_attack_processed.xlsx')
df = df[["id", "cmt_processed", "attack_type"]]
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3693 entries, 0 to 3692
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id             3693 non-null   int64 
 1   cmt_processed  3693 non-null   object
 2   attack_type    3693 non-null   object
dtypes: int64(1), object(2)
memory usage: 86.7+ KB


PhoBERT

In [31]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

# ==========================================
# CẤU HÌNH NHÃN (LABEL MAP)
# ==========================================
LABEL_MAP = [
    "Threat", "Scam", "Misinformation", "Boycott",
    "Body Shaming", "Sexual Harassment", "Intelligence", "Moral", "Victim Blaming",
    "Gender", "Regionalism", "Racism", "Classism", "Religion",
    "Politics", "Social Issues", "Product", "Community",
    "Other"
]

# ==========================================
# XỬ LÝ NHÃN (ONE-HOT ENCODING)
# ==========================================

# Bước 1: Chuyển chuỗi "Body Shaming, Moral" thành List ["Body Shaming", "Moral"]
def process_tags(text):
    if not isinstance(text, str):
        return []
    return [t.strip() for t in text.split(',') if t.strip() != '']

df['tags_list'] = df['attack_type'].apply(process_tags)

# Bước 2: One-Hot Encoding theo LABEL_MAP cố định
mlb = MultiLabelBinarizer(classes=LABEL_MAP)
onehot_labels = mlb.fit_transform(df['tags_list'])

print("Kích thước one-hot matrix:", onehot_labels.shape)

print("\n--- Kiểm tra mẫu đầu tiên ---")
print("Gốc:", df['attack_type'].iloc[0])
print("List:", df['tags_list'].iloc[0])
print("One-hot:", onehot_labels[0])
print("Mapping lại:", mlb.inverse_transform(onehot_labels[0:1]))


Kích thước one-hot matrix: (3693, 19)

--- Kiểm tra mẫu đầu tiên ---
Gốc: Scam, Moral, Threat
List: ['Scam', 'Moral', 'Threat']
One-hot: [1 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0]
Mapping lại: [('Threat', 'Scam', 'Moral')]


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['Other (AI Failed)'] will be ignored
  warnings.warn(


In [32]:
!pip install iterative-stratification

In [33]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
# Thư viện chia dữ liệu chuyên dụng cho Multi-label
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# ==========================================
# CHIA TẬP DỮ LIỆU
# ==========================================

# Chuyển dữ liệu sang numpy array để index dễ dàng hơn
X = df['cmt_processed'].to_numpy()
y = onehot_labels

# Bước 1: Chia Train (80%) - Temp (20%)
msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_index, temp_index in msss.split(X, y):
    X_train, X_temp = X[train_index], X[temp_index]
    y_train, y_temp = y[train_index], y[temp_index]

# Bước 2: Chia Temp thành Dev (50% của Temp) - Test (50% của Temp)
msss_val = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)

for dev_index, test_index in msss_val.split(X_temp, y_temp):
    X_dev, X_test = X_temp[dev_index], X_temp[test_index]
    y_dev, y_test = y_temp[dev_index], y_temp[test_index]

print(f"\nSố lượng Train: {len(X_train)}")
print(f"Số lượng Dev:   {len(X_dev)}")
print(f"Số lượng Test:  {len(X_test)}")

# ==========================================
# TẠO DATASET CHO PYTORCH
# ==========================================
class AttackTypeDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.float)

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': label
        }

# Khởi tạo Tokenizer
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")

train_dataset = AttackTypeDataset(X_train, y_train, tokenizer)
dev_dataset = AttackTypeDataset(X_dev, y_dev, tokenizer)
test_dataset = AttackTypeDataset(X_test, y_test, tokenizer)

# Tạo DataLoader
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)


Số lượng Train: 2954
Số lượng Dev:   369
Số lượng Test:  370


In [34]:
import torch.nn as nn
from transformers import AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, hamming_loss
from tqdm.auto import tqdm
import os

# Cấu hình thiết bị
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Khai báo mô hình
class PhoBertForMultiLabel(nn.Module):
    def __init__(self, n_classes, model_name="vinai/phobert-base"):
        super(PhoBertForMultiLabel, self).__init__()
        self.phobert = AutoModel.from_pretrained(model_name)
        self.drop = nn.Dropout(p=0.3) # Dropout để tránh Overfitting
        self.out = nn.Linear(768, n_classes) # 768 là hidden size của PhoBERT base

    def forward(self, input_ids, attention_mask):
        # Lấy output từ PhoBERT
        # pooled_output: đại diện cho vector của token [CLS] (toàn câu)
        outputs = self.phobert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        pooled_output = outputs[1]

        output = self.drop(pooled_output)
        return self.out(output) # Trả về logits (chưa qua Sigmoid)

# Khởi tạo mô hình
model = PhoBertForMultiLabel(n_classes=len(LABEL_MAP))
model = model.to(device)

Using device: cuda


In [35]:
import numpy as np
import torch
import os

class EarlyStopping:
    def __init__(self, patience=5, verbose=False, delta=0, path='best_phobert_attack_type.bin', trace_func=print):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_score_max = -np.inf # Khởi tạo -Vô cùng vì ta đang tìm Max F1
        self.delta = delta
        self.path = path
        self.trace_func = trace_func

    def __call__(self, score, model):
        # score ở đây là F1-Micro

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(score, model)
        elif score < self.best_score + self.delta:
            # Nếu điểm mới KHÔNG tốt hơn điểm cũ (cộng thêm delta)
            self.counter += 1
            if self.verbose:
                self.trace_func(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            # Nếu điểm mới tốt hơn
            self.best_score = score
            self.save_checkpoint(score, model)
            self.counter = 0

    def save_checkpoint(self, score, model):
        '''Lưu model khi validation score cải thiện.'''
        if self.verbose:
            self.trace_func(f'F1-Score cải thiện ({self.val_score_max:.4f} --> {score:.4f}). Đang lưu model...')
        torch.save(model.state_dict(), self.path)
        self.val_score_max = score

In [36]:
# Hyperparameters
EPOCHS = 30
LEARNING_RATE = 2e-5
THRESHOLD = 0.3
PATIENCE = 5

# Loss function: BCEWithLogitsLoss
criterion = nn.BCEWithLogitsLoss()

# Optimizer: AdamW
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

# Scheduler: Giảm dần learning rate tuyến tính
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# Khởi tạo Early Stopping
early_stopping = EarlyStopping(patience=PATIENCE, verbose=True, path='best_phobert_attack_type.bin')

In [37]:
def train_epoch(model, data_loader, loss_fn, optimizer, device, scheduler, n_examples):
    model = model.train()
    losses = []

    # Progress bar
    progress_bar = tqdm(data_loader, desc="Training", leave=False)

    for d in progress_bar:
        input_ids = d["input_ids"].to(device)
        attention_mask = d["attention_mask"].to(device)
        targets = d["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = loss_fn(outputs, targets)
        losses.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # Tránh bùng nổ gradient
        optimizer.step()
        scheduler.step()

        # Update progress bar
        progress_bar.set_postfix(loss=loss.item())

    return np.mean(losses)

def eval_model(model, data_loader, loss_fn, device, n_examples, threshold=0.35):
    model = model.eval()
    losses = []
    fin_targets = []
    fin_outputs = []

    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            targets = d["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs, targets)
            losses.append(loss.item())

            # Chuyển logits -> xác suất
            probs = torch.sigmoid(outputs).cpu().detach().numpy()

            # SỬ DỤNG THAM SỐ threshold ĐƯỢC TRUYỀN VÀO
            preds = (probs >= threshold).astype(int)

            fin_targets.extend(targets.cpu().detach().numpy())
            fin_outputs.extend(preds)

    avg_loss = np.mean(losses)

    # Tính các chỉ số đánh giá
    f1_micro = f1_score(fin_targets, fin_outputs, average='micro')
    f1_macro = f1_score(fin_targets, fin_outputs, average='macro')
    hamming = hamming_loss(fin_targets, fin_outputs)

    return avg_loss, f1_micro, f1_macro, hamming, fin_targets, fin_outputs

In [38]:
history = {'train_loss': [], 'val_loss': [], 'val_f1': []}
print(f"BẮT ĐẦU HUẤN LUYỆN (Max Epochs: {EPOCHS}, Patience: {PATIENCE})")
print(f"{'Epoch':^5} | {'Train Loss':^10} | {'Val Loss':^10} | {'Val F1 (Micro)':^15} | {'Hamming':^10}")
print("-" * 65)

for epoch in range(EPOCHS):
    train_loss = train_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device,
        scheduler,
        len(X_train)
    )

    val_loss, val_f1_micro, val_f1_macro, val_hamming, _, _ = eval_model(
        model,
        dev_loader,
        criterion,
        device,
        len(X_dev),
        threshold=THRESHOLD
    )

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_f1'].append(val_f1_micro)

    print(f"{epoch + 1:^5} | {train_loss:^10.4f} | {val_loss:^10.4f} | {val_f1_micro:^15.4f} | {val_hamming:^10.4f}")

    early_stopping(val_f1_micro, model)

    if early_stopping.early_stop:
        print("\n Dừng sớm vì model không cải thiện nữa!")
        break

print("\nĐã hoàn thành huấn luyện.")

model.load_state_dict(torch.load('best_phobert_attack_type.bin'))
print("Đã load lại trọng số (weights) tốt nhất vào model.")

BẮT ĐẦU HUẤN LUYỆN (Max Epochs: 30, Patience: 5)
Epoch | Train Loss |  Val Loss  | Val F1 (Micro)  |  Hamming  
-----------------------------------------------------------------


Training:   0%|          | 0/185 [00:00<?, ?it/s]

  1   |   0.3239   |   0.2319   |     0.2845      |   0.0940  
F1-Score cải thiện (-inf --> 0.2845). Đang lưu model...


Training:   0%|          | 0/185 [00:00<?, ?it/s]

  2   |   0.2325   |   0.2199   |     0.2753      |   0.0849  
EarlyStopping counter: 1 out of 5


...

 21   |   0.0631   |   0.1790   |     0.5897      |   0.0639  
EarlyStopping counter: 3 out of 5


Training:   0%|          | 0/185 [00:00<?, ?it/s]

 22   |   0.0606   |   0.1796   |     0.5756      |   0.0660  
EarlyStopping counter: 4 out of 5


Training:   0%|          | 0/185 [00:00<?, ?it/s]

 23   |   0.0581   |   0.1792   |     0.5902      |   0.0642  
EarlyStopping counter: 5 out of 5

 Dừng sớm vì model không cải thiện nữa!

Đã hoàn thành huấn luyện.
Đã load lại trọng số (weights) tốt nhất vào model.


In [39]:
# --- ĐÁNH GIÁ TRÊN TẬP TEST ---
from sklearn.metrics import classification_report
print("\n--- KẾT QUẢ TRÊN TẬP TEST ---")

# Load lại weight tốt nhất
model.load_state_dict(torch.load('best_phobert_attack_type.bin'))

test_loss, test_f1_micro, test_f1_macro, test_hamming, y_true, y_pred = eval_model(
    model,
    test_loader,
    criterion,
    device,
    len(X_test)
)

print(f"Test F1 (Micro): {test_f1_micro:.4f}")
print(f"Test Hamming Loss: {test_hamming:.4f}")

# IN BÁO CÁO CHI TIẾT TỪNG NHÃN
print("\n--- CHI TIẾT TỪNG LOẠI TẤN CÔNG ---")
print(classification_report(
    y_true,
    y_pred,
    target_names=LABEL_MAP,
    zero_division=0
))

# --- HÀM DỰ ĐOÁN THỰC TẾ ---
def predict_attack_type(text, model, tokenizer, label_map, threshold= THRESHOLD):
    model.eval()
    encoding = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=128,
        return_token_type_ids=False,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt',
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask)
        # Sigmoid để ra xác suất
        probs = torch.sigmoid(outputs).cpu().numpy()[0]

    # Lấy các nhãn vượt qua ngưỡng
    results = []
    for idx, prob in enumerate(probs):
        if prob >= threshold:
            results.append((label_map[idx], prob))

    return results


--- KẾT QUẢ TRÊN TẬP TEST ---
Test F1 (Micro): 0.6011
Test Hamming Loss: 0.0600

--- CHI TIẾT TỪNG LOẠI TẤN CÔNG ---
                   precision    recall  f1-score   support

           Threat       0.60      0.69      0.64        36
             Scam       0.14      0.09      0.11        11
   Misinformation       0.00      0.00      0.00         8
          Boycott       0.00      0.00      0.00         4
     Body Shaming       0.61      0.50      0.55        28
Sexual Harassment       0.34      0.33      0.34        33
     Intelligence       0.83      0.74      0.78       102
            Moral       0.62      0.74      0.67       131
   Victim Blaming       0.00      0.00      0.00         4
           Gender       0.60      0.17      0.26        18
      Regionalism       0.70      0.54      0.61        13
           Racism       0.54      0.47      0.50        15
         Classism       0.33      0.08      0.13        12
         Religion       0.00      0.00      0.00       

In [40]:
from sklearn.metrics import classification_report

# 1. HÀM LẤY XÁC SUẤT GỐC
def get_predictions_probs(model, data_loader, device):
    model = model.eval()
    fin_targets = []
    fin_probs = [] # Lưu xác suất thực

    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            targets = d["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            # Chỉ lấy xác suất, chưa chia ngưỡng
            probs = torch.sigmoid(outputs).cpu().detach().numpy()

            fin_targets.extend(targets.cpu().detach().numpy())
            fin_probs.extend(probs)

    return np.array(fin_targets), np.array(fin_probs)

# Load lại weight tốt nhất
model.load_state_dict(torch.load('best_phobert_attack_type.bin'))

print("Đang chạy dự đoán trên tập Test...")
y_true, y_probs = get_predictions_probs(model, test_loader, device)

# 2. HÀM IN BÁO CÁO THEO NGƯỠNG
def report_with_threshold(threshold):
    print(f"\n{'='*20} KẾT QUẢ VỚI THRESHOLD = {threshold} {'='*20}")
    y_pred = (y_probs >= threshold).astype(int)

    # Tính toán
    micro = f1_score(y_true, y_pred, average='micro')
    macro = f1_score(y_true, y_pred, average='macro')
    hamming = hamming_loss(y_true, y_pred)

    print(f"Test F1 (Micro): {micro:.4f}")
    print(f"Test F1 (Macro): {macro:.4f}")
    print(f"Test Hamming Loss: {hamming:.4f}")

    print("\n--- CHI TIẾT TỪNG NHÃN ---")
    print(classification_report(y_true, y_pred, target_names=LABEL_MAP, zero_division=0))
    return y_pred # Trả về để dùng in mẫu

# --- THỬ NGHIỆM ---
y_pred = report_with_threshold(0.3)


# --- HÀM DỰ ĐOÁN THỰC TẾ ---
def predict_attack_type(text, model, tokenizer, label_map, threshold= THRESHOLD):
    model.eval()
    encoding = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=128,
        return_token_type_ids=False,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt',
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask)
        # Sigmoid để ra xác suất
        probs = torch.sigmoid(outputs).cpu().numpy()[0]

    # Lấy các nhãn vượt qua ngưỡng
    results = []
    for idx, prob in enumerate(probs):
        if prob >= threshold:
            results.append((label_map[idx], prob))

    return results

Đang chạy dự đoán trên tập Test...

==================== KẾT QUẢ VỚI THRESHOLD = 0.3 ====================
Test F1 (Micro): 0.6042
Test F1 (Macro): 0.3455
Test Hamming Loss: 0.0613

--- CHI TIẾT TỪNG NHÃN ---
                   precision    recall  f1-score   support

           Threat       0.57      0.69      0.62        36
             Scam       0.25      0.18      0.21        11
   Misinformation       0.00      0.00      0.00         8
          Boycott       0.00      0.00      0.00         4
     Body Shaming       0.62      0.54      0.58        28
Sexual Harassment       0.32      0.33      0.33        33
     Intelligence       0.83      0.75      0.79       102
            Moral       0.61      0.75      0.67       131
   Victim Blaming       0.00      0.00      0.00         4
           Gender       0.40      0.22      0.29        18
      Regionalism       0.54      0.54      0.54        13
           Racism       0.50      0.53      0.52        15
         Classism       

In [41]:
# Thử nghiệm một câu
sample_cmt = "lũ hồi_giáo mọi_rợ chết_sạch đi"
print("\n--- DỰ ĐOÁN MẪU ---")
print(f"Input: {sample_cmt}")
preds = predict_attack_type(sample_cmt, model, tokenizer, LABEL_MAP, threshold= 0.35)
print("Kết quả:", preds)


--- DỰ ĐOÁN MẪU ---
Input: lũ hồi_giáo mọi_rợ chết_sạch đi
Kết quả: [('Threat', np.float32(0.8903585)), ('Politics', np.float32(0.8836018))]


In [42]:
def print_test_predictions(X_test_data, y_true_bin, y_pred_bin, label_map, num_samples=10):
    print(f"\n{'='*20} DỰ ĐOÁN MẪU TRÊN TẬP TEST {'='*20}")

    # Hàm helper chuyển binary [0, 1, 0] sang tên nhãn ['Label B']
    def decode_labels(binary_row):
        return [label_map[i] for i, val in enumerate(binary_row) if val == 1]

    indices = np.random.choice(len(X_test_data), size=min(num_samples, len(X_test_data)), replace=False)

    for idx in indices:
        text = X_test_data[idx]

        if hasattr(X_test_data, 'iloc'):
             text = X_test_data.iloc[idx]

        true_labels = decode_labels(y_true_bin[idx])
        pred_labels = decode_labels(y_pred_bin[idx])

        print(f"ID: {idx}")
        print(f"   CMT: \"{text}\"")
        print(f"   Thực tế: {true_labels}")

        is_correct = set(true_labels) == set(pred_labels)
        status = "ĐÚNG" if is_correct else "SAI/THIẾU"

        print(f"  Dự đoán: {pred_labels} -> {status}")
        print("-" * 60)

# X_test: List các comment gốc
# y_true: List các nhãn thực tế (dạng binary list)
# y_pred: List các nhãn dự đoán (dạng binary list)
print_test_predictions(list(X_test), y_true, y_pred, LABEL_MAP, num_samples=10)


==================== DỰ ĐOÁN MẪU TRÊN TẬP TEST ====================
ID: 142
   CMT: "thứ không nhân_cách , tiền nhìu cko người ta là cai_nghiệp kiep truoc thiếu người ta giờ trả chứ méo được miếng_đức nào đâu mà hô_hào , chị ba ! chị ba phi"
   Thực tế: ['Moral']
  Dự đoán: ['Moral'] -> ĐÚNG
------------------------------------------------------------
ID: 361
   CMT: "dm ... thằng assmin phản_lại cái_động ... gác máy cái mả cụ mày ... sao mày không nghĩ dây dân_tộc bị cá_mập cắn ... dm ..."
   Thực tế: ['Moral']
  Dự đoán: ['Politics'] -> SAI/THIẾU
------------------------------------------------------------
ID: 276
   CMT: "có lửa mới có khói . cũng như vụ tiên_lãng hải_phòng mà thôi . chết oan cho bọn 🐕 tham_lam ăn cướp"
   Thực tế: ['Moral', 'Victim Blaming']
  Dự đoán: ['Threat', 'Moral'] -> SAI/THIẾU
------------------------------------------------------------
ID: 149
   CMT: "bả chửi người ta lên mạng câu like , bả mấy là người câu like đó , kêu người này người kia ủng_hộ nhiều 

In [43]:
!pip install onnx onnxscript

In [44]:
# --- CODE EXPORT ONNX CHUẨN (DYNAMIC BATCH) ---
import torch

# 1. Load model Pytorch (Đảm bảo model đã load weights ngon lành)
# Giả sử biến 'model' là model PhoBERT Type Attack của bạn
model.eval()
model.to("cpu")

# 2. Tạo input giả lập (Batch size = 1, Length = 128)
dummy_input_ids = torch.randint(0, 1000, (1, 128), dtype=torch.long)
dummy_attn_mask = torch.ones((1, 128), dtype=torch.long)

# 3. Export
output_path = "model_dynamic.onnx" # Lưu tên mới để phân biệt

torch.onnx.export(
    model,
    (dummy_input_ids, dummy_attn_mask),
    output_path,
    export_params=True,
    opset_version=12, # Khuyến nghị version 12 trở lên cho Transformer
    do_constant_folding=True,
    input_names=['input_ids', 'attention_mask'],
    output_names=['output'],
    dynamic_axes={
        'input_ids': {0: 'batch_size'},      # <--- QUAN TRỌNG: Cho phép batch thay đổi
        'attention_mask': {0: 'batch_size'}, # <--- QUAN TRỌNG
        'output': {0: 'batch_size'}          # <--- QUAN TRỌNG
    }
)

print(f"✅ Đã export model ONNX hỗ trợ Batch tại: {output_path}")

/tmp/ipython-input-2759851045.py:16: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0122 05:50:23.796000 440 torch/onnx/_internal/exporter/_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 12 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `PhoBertForMultiLabel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `PhoBertForMultiLabel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...


[torch.onnx] Translate the graph into ONNX... ✅


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 127, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 122, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /github/workspace/onnx/version_converter/adapters/no_previous_version.h:26: adapt: Assertion `

Applied 54 of general pattern rewrite rules.
✅ Đã export model ONNX hỗ trợ Batch tại: model_dynamic.onnx


In [45]:
import torch
import os
import shutil

# --- CẤU HÌNH ---
OUTPUT_DIR = "/content/drive/MyDrive/SE363-Team13/model/type_attack_0.3"
ONNX_MODEL_NAME = "model.onnx"
ZIP_NAME = "phobert_model_package"

# 1. Tạo thư mục sạch
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR)


# 2. Lưu Tokenizer (Rất quan trọng cho Spark)
# Giả sử biến 'tokenizer' là cái bạn đã dùng lúc train
tokenizer.save_pretrained(OUTPUT_DIR)
print("Đã lưu Tokenizer (vocab, config)")

# 3. Lưu file Config của PhoBERT (để load lại kiến trúc nếu cần)
model.phobert.config.save_pretrained(OUTPUT_DIR)

# 4. Xuất mô hình sang chuẩn ONNX
model.eval() # Chuyển về chế độ inference

# Tạo dữ liệu giả để định hình mạng
dummy_text = "Thử nghiệm xuất mô hình"
dummy_input = tokenizer.encode_plus(
    dummy_text,
    max_length=128,
    padding='max_length',
    truncation=True,
    return_tensors='pt'
)
device = next(model.parameters()).device #
input_ids = dummy_input['input_ids'].to(device)
attention_mask = dummy_input['attention_mask'].to(device)

# Thực hiện export
onnx_path = os.path.join(OUTPUT_DIR, ONNX_MODEL_NAME)
try:
    torch.onnx.export(
        model,
        (input_ids, attention_mask),                # Đầu vào mẫu
        onnx_path,                                  # Đường dẫn xuất
        export_params=True,                         # Lưu kèm trọng số đã train
        opset_version=18,                           # Phiên bản ONNX ổn định
        do_constant_folding=True,                   # Tối ưu hóa các hằng số
        input_names=['input_ids', 'attention_mask'],# Tên đầu vào
        output_names=['output'],                    # Tên đầu ra
        dynamic_axes={                              # Cho phép batch size thay đổi linh hoạt
            'input_ids': {0: 'batch_size'},
            'attention_mask': {0: 'batch_size'},
            'output': {0: 'batch_size'}
        }
    )
    print(f" Đã export thành công: {onnx_path}")
except Exception as e:
    print(f"Lỗi export ONNX: {e}")

# 5. Đóng gói thành file ZIP
shutil.make_archive(ZIP_NAME, 'zip', OUTPUT_DIR)
print(f"HOÀN TẤT!{ZIP_NAME}.zip")

Đã lưu Tokenizer (vocab, config)


/tmp/ipython-input-3119750456.py:43: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `PhoBertForMultiLabel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `PhoBertForMultiLabel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 54 of general pattern rewrite rules.
 Đã export thành công: /content/drive/MyDrive/SE363-Team13/model/type_attack_0.3/model.onnx
HOÀN TẤT!phobert_model_package.zip


In [46]:
import numpy as np
from sklearn.metrics import f1_score, classification_report
import torch

# ============================================================
# HÀM DÒ TÌM NGƯỠNG TỐI ƯU (THRESHOLD TUNING)
# ============================================================
def find_best_threshold(model, data_loader, device):
    model.eval()
    all_targets = []
    all_probs = []

    print("⏳ Đang quét tập Test để lấy xác suất dự đoán...")

    # 1. Lấy toàn bộ xác suất (Probs) chưa qua xử lý
    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            targets = d["labels"].to(device)

            outputs = model(input_ids, attention_mask)
            # Chuyển Logits -> Sigmoid (Xác suất 0-1)
            probs = torch.sigmoid(outputs).cpu().numpy()

            all_probs.extend(probs)
            all_targets.extend(targets.cpu().numpy())

    all_probs = np.array(all_probs)
    all_targets = np.array(all_targets)

    # 2. Vòng lặp thử các ngưỡng từ 0.1 đến 0.6
    best_thresh = 0.5
    best_f1 = 0.0

    print(f"\n{'Threshold':^10} | {'F1 Micro':^10} | {'Ghi chú'}")
    print("-" * 45)

    # Quét từ 0.15 đến 0.60, bước nhảy 0.01 (Chi tiết từng %)
    for thresh in np.arange(0.15, 0.60, 0.01):
        y_pred = (all_probs >= thresh).astype(int)
        f1 = f1_score(all_targets, y_pred, average='micro')

        note = ""
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh
            note = "⭐ Đỉnh mới"

        # Chỉ in ra dòng nào có F1 > 0.5 để đỡ rối mắt
        if f1 > 0.5:
            print(f"{thresh:^10.2f} | {f1:^10.4f} | {note}")

    print("-" * 45)
    print(f"🏆 NGƯỠNG TỐT NHẤT: {best_thresh:.2f}")
    print(f"🔥 F1 Micro cao nhất: {best_f1:.4f}")

    return best_thresh, all_targets, all_probs

# ============================================================
# CHẠY THỰC TẾ
# ============================================================

# 1. Đảm bảo model đang là model tốt nhất (cách cũ)
model.load_state_dict(torch.load('best_phobert_attack_type.bin', map_location=device))

# 2. Tìm ngưỡng
best_threshold, y_true, y_probs = find_best_threshold(model, test_loader, device)

# 3. In báo cáo chi tiết với ngưỡng vừa tìm được
print(f"\n--- BÁO CÁO CHI TIẾT VỚI NGƯỠNG {best_threshold:.2f} ---")
y_pred_optimal = (y_probs >= best_threshold).astype(int)

print(classification_report(
    y_true,
    y_pred_optimal,
    target_names=LABEL_MAP,
    zero_division=0
))

⏳ Đang quét tập Test để lấy xác suất dự đoán...

Threshold  |  F1 Micro  | Ghi chú
---------------------------------------------
   0.15    |   0.5829   | ⭐ Đỉnh mới
   0.16    |   0.5885   | ⭐ Đỉnh mới
   0.17    |   0.5932   | ⭐ Đỉnh mới
   0.18    |   0.5938   | ⭐ Đỉnh mới
   0.19    |   0.6002   | ⭐ Đỉnh mới
   0.20    |   0.6015   | ⭐ Đỉnh mới
   0.21    |   0.6043   | ⭐ Đỉnh mới
   0.22    |   0.6046   | ⭐ Đỉnh mới
   0.23    |   0.6033   | 
   0.24    |   0.5998   | 
   0.25    |   0.6014   | 
   0.26    |   0.6043   | 
   0.27    |   0.6018   | 
   0.28    |   0.5995   | 
   0.29    |   0.6033   | 
   0.30    |   0.6042   | 
   0.31    |   0.6013   | 
   0.32    |   0.6039   | 
   0.33    |   0.6060   | ⭐ Đỉnh mới
   0.34    |   0.6026   | 
   0.35    |   0.6011   | 
   0.36    |   0.6038   | 
   0.37    |   0.6034   | 
   0.38    |   0.6031   | 
   0.39    |   0.6000   | 
   0.40    |   0.5943   | 
   0.41    |   0.5982   | 
   0.42    |   0.5990   | 
   0.43    |   0.6014   |

In [47]:
import torch
import os

# --- CẤU HÌNH ---
INPUT_BIN_PATH = "best_phobert_attack_type.bin"  # Tên file .bin hiện có của bạn
OUTPUT_PTH_PATH = "model_2.pth"      # Tên file .pth muốn lưu

# --- XỬ LÝ ---
print(f"⏳ Đang load file: {INPUT_BIN_PATH} ...")

if os.path.exists(INPUT_BIN_PATH):
    try:
        # 1. Load weights từ file .bin
        # map_location='cpu' để đảm bảo chạy được kể cả khi không có GPU
        state_dict = torch.load(INPUT_BIN_PATH, map_location=torch.device('cpu'))

        # 2. Lưu lại dưới dạng .pth
        torch.save(state_dict, OUTPUT_PTH_PATH)

        print(f"✅ CHUYỂN ĐỔI THÀNH CÔNG!")
        print(f"📂 File mới: {OUTPUT_PTH_PATH}")
        print("👉 Hãy tải file này về và bỏ vào thư mục: saved_models/type_attack/")

    except Exception as e:
        print(f"❌ Lỗi khi đọc file: {e}")
else:
    print(f"❌ Không tìm thấy file '{INPUT_BIN_PATH}'. Vui lòng kiểm tra lại đường dẫn.")

⏳ Đang load file: best_phobert_attack_type.bin ...
✅ CHUYỂN ĐỔI THÀNH CÔNG!
📂 File mới: model_2.pth
👉 Hãy tải file này về và bỏ vào thư mục: saved_models/type_attack/
